In [ ]:
#| default_exp preprocessing

# Preprocessing
> Utilities for processing data before training/analysis

In [ ]:
#| export
import warnings
from typing import Union

import numpy as np
import pandas as pd

In [ ]:
#| hide
from nbdev import show_doc

from utilsforecast.data import generate_series

In [ ]:
#| exporti
def _determine_bound(bound, freq, times_by_id, agg) -> np.ndarray:
    if bound == 'per_serie':
        out = times_by_id[agg].values
    else:
        # the following return a scalar
        if bound == 'global':
            val = getattr(times_by_id[agg].values, agg)()
            if isinstance(freq, str):
                val = np.datetime64(val)
        else:
            if isinstance(freq, str):
                # this raises a nice error message if it isn't a valid datetime
                val = np.datetime64(bound)
            else:
                val = bound
        out = np.full(times_by_id.shape[0], val)
    if isinstance(freq, str):
        out = out.astype(f'datetime64[{freq}]')
    return out

In [ ]:
#| export
def fill_gaps(
    df: pd.DataFrame,
    freq: Union[str, int],
    start: Union[str, int] = 'per_serie',
    end: Union[str, int] = 'global',
    id_col: str = 'unique_id',
    time_col: str = 'ds',
) -> pd.DataFrame:
    """Enforce start and end datetimes for dataframe.

    Parameters
    ----------
    df : pandas DataFrame
        Input data
    freq : str or int
        Series' frequency
    start : str
        Initial timestamp for the series.
            * 'per_serie' uses each serie's first timestamp
            * 'global' uses the first timestamp seen in the data
            * Can also be a specific timestamp or integer, e.g. '2000-01-01' or 2000
    end : str
        Initial timestamp for the series.
            * 'per_serie' uses each serie's last timestamp
            * 'global' uses the last timestamp seen in the data
            * Can also be a specific timestamp or integer, e.g. '2000-01-01' or 2000
    id_col : str (default='unique_id')
        Column that identifies each serie.
    time_col : str (default='ds')
        Column that identifies each timestamp.

    Returns
    -------
    filled_df : pandas DataFrame
        Dataframe with gaps filled.
    """
    if isinstance(freq, str):
        offset = pd.tseries.frequencies.to_offset(freq)
        if not hasattr(offset, 'delta'):
            # irregular freq, try using first letter of abbreviation
            # such as MS = 'Month Start' -> 'M', YS = 'Year Start' -> 'Y'
            freq = freq[0]
        delta: Union[np.timedelta64, int] = np.timedelta64(1, freq)
    else:
        delta = 1
    times_by_id = df.groupby(id_col, observed=True)[time_col].agg(['min', 'max'])
    starts = _determine_bound(start, freq, times_by_id, 'min')
    ends = _determine_bound(end, freq, times_by_id, 'max') + delta
    sizes = ((ends - starts) / delta).astype(np.int64)
    times = pd.Index(
        np.concatenate(
            [
                np.arange(start, end, delta) for start, end in zip(starts, ends)
            ]
        )
    )
    if isinstance(freq, str):
        times = times.astype('datetime64[ns]', copy=False)
        first_time = np.datetime64(df.iloc[0][time_col])
        was_truncated = first_time != first_time.astype(f'datetime64[{freq}]')
        if was_truncated:
            times += offset
    uids = np.repeat(times_by_id.index, sizes)
    idx = pd.MultiIndex.from_arrays([uids, times], names=[id_col, time_col])
    res = df.set_index([id_col, time_col]).reindex(idx).reset_index()
    extra_cols = df.columns.drop([id_col, time_col]).tolist()
    if extra_cols:
        check_col = extra_cols[0]
        if res[check_col].count() < df[check_col].count():
            warnings.warn(
                "Some values were lost during filling, "
                "please make sure that all your times meet the specified frequency.\n"
                "For example if you have 'W-TUE' as your frequency, "
                "make sure that all your times are actually Tuesdays."
            )
    return res

In [ ]:
show_doc(fill_gaps)

In [ ]:
df = pd.DataFrame(
    {
        'unique_id': [0, 0, 0, 1, 1],
        'ds': pd.to_datetime(['2020', '2021', '2023', '2021', '2022']),
        'y': np.arange(5),
    }
)
df

The default functionality is taking the current starts and only extending the end date to be the same for all series.

In [ ]:
fill_gaps(
    df,
    freq='YS',
)

We can also specify `end='per_serie'` to only fill possible gaps within each serie.

In [ ]:
fill_gaps(
    df,
    freq='YS',
    end='per_serie',
)

We can also specify an end date in the future.

In [ ]:
fill_gaps(
    df,
    freq='YS',
    end='2024',
)

We can set all series to start at the same time.

In [ ]:
fill_gaps(
    df,
    freq='YS',
    start='global'
)

We can also set a common start date for all series (which can be earlier than their current starts).

In [ ]:
fill_gaps(
    df,
    freq='YS',
    start='2019',
)

In case the times are integers the frequency, start and end must also be integers.

In [ ]:
df = pd.DataFrame(
    {
        'unique_id': [0, 0, 0, 1, 1],
        'ds': [2020, 2021, 2023, 2021, 2022],
        'y': np.arange(5),
    }
)
df

In [ ]:
fill_gaps(
    df,
    freq=1,
    start=2019,
    end=2024,
)

In [ ]:
#| hide
from itertools import product

from fastcore.test import test_warns

In [ ]:
#| hide
def check_fill(dates, freq, start, end, include_start, include_end):    
    min_idx = 1 if include_start else 2
    max_idx = dates.size if include_end else dates.size - 1
    date_idxs = np.hstack(
        [
            np.random.choice(
                np.arange(min_idx, max_idx),
                size=n_periods // 2,
                replace=False
            ) for _ in range(2)
        ]
    )
    data = pd.DataFrame(
        {
            'unique_id': np.repeat([1, 2], n_periods // 2),
            'ds': dates[date_idxs],
            'y': np.arange(n_periods, dtype=np.float64)
        }
    )
    filled = fill_gaps(data, freq, start=start, end=end)
    data_starts_ends = data.groupby('unique_id', observed=True)['ds'].agg(['min', 'max'])
    global_start = data_starts_ends['min'].min()
    global_end = data_starts_ends['max'].max()
    filled_starts_ends = filled.groupby('unique_id', observed=True)['ds'].agg(['min', 'max'])
    
    # fill keeps original data
    assert filled['y'].count() == n_periods
    # check starts
    if start == 'per_serie':
        pd.testing.assert_series_equal(
            data_starts_ends['min'], filled_starts_ends['min'],
        )        
    else: # global or specific
        min_dates = filled_starts_ends['min'].unique()
        assert min_dates.size == 1
        expected_start = global_start if start == 'global' else start
        assert min_dates[0] == expected_start

    # check ends
    if end == 'per_serie':
        pd.testing.assert_series_equal(
            data_starts_ends['max'], filled_starts_ends['max'],
        )        
    else: # global or specific
        max_dates = filled_starts_ends['max'].unique()
        assert max_dates.size == 1
        expected_end = global_end if end == 'global' else end
        assert max_dates[0] == expected_end


n_periods = 100
for freq in ['Y', 'YS', 'M', 'MS', 'W', 'W-TUE', 'D', 's', 'ms', 1]:
    if isinstance(freq, str):         
        dates = pd.date_range('1900-01-01', periods=n_periods, freq=freq)
        offset = pd.tseries.frequencies.to_offset(freq)
    else:
        dates = np.arange(n_periods, dtype=np.int64)
        offset = 1
    global_start = dates[0]
    global_end = dates[-1]
    starts = ['global', 'per_serie', global_start - offset]
    ends = ['global', 'per_serie', global_end + offset]
    include_starts = [True, False]
    include_ends = [True, False]
    iterable = product(starts, ends, include_starts, include_ends)
    for start, end, include_start, include_end in iterable:
        check_fill(dates, freq, start, end, include_start, include_end)

In [ ]:
#| hide
# last value doesn't meet frequency (year start)
dfx = pd.DataFrame(
    {
        'unique_id': [0, 0, 0, 1, 1],
        'ds': pd.to_datetime(['2020-01', '2021-01', '2023-01', '2021-01', '2022-02']),
        'y': np.arange(5),
    }
)
with warnings.catch_warnings(record=True) as w:
    fill_gaps(dfx, 'YS')
assert 'values were lost' in str(w[0].message)